# GBT-05 independent test

In [1]:
import csv
import json
from datetime import datetime, timezone
from pathlib import Path
from pyspark import StorageLevel
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql import SparkSession, functions as F

THRESHOLDS = [0.0, 1.0, 5.0, 10.0]
RAINFALL_COLUMNS = ['rainfall_previous_5m', 'rainfall_previous_30m', 'rainfall_previous_60m', 'future_rainfall_30m']
REQUIRED_COLUMNS = ['station_id', 'prediction_time', 'hour', 'month', *RAINFALL_COLUMNS]


In [2]:
# Run on a Vanda compute node using the environment used for training.
# Set MODEL_ROOT if you open this notebook outside the project checkout.
here = Path.cwd().resolve()
MODEL_ROOT = next((p for base in [here, *here.parents]
                   for p in (base, base / 'task2' / 'Model_GBT')
                   if (p / 'outputs' / 'tuning_stage1').is_dir()), None)
# Optional explicit paths, e.g. Path('/nfs/home/svu/le.pan/.../Model_GBT'):
RUN_ROOT = None  # Exact frozen run directory if automatic lookup is ambiguous.
CORES = 4
DRIVER_MEMORY = '16g'
DECISION_THRESHOLD = 0.5  # Fix before testing; do not tune using test results.


## Evaluation helpers
These cells only define functions; the test starts in the execution cell below.

In [3]:
def evaluate(predictions, split):
    """Summarize ranking, probability error and reliability for each threshold."""
    scored = predictions.withColumn(
        'p', vector_to_array('probability')[1]
    ).persist(StorageLevel.DISK_ONLY)
    rows = []
    try:
        for threshold in THRESHOLDS:
            part = scored.where(F.col('threshold_mm') == threshold)
            summary = part.agg(
                F.count('*').alias('samples'),
                F.sum('label').alias('positives'),
                F.avg(F.pow(F.col('p') - F.col('label'), 2)).alias('brier')
            ).first()
            if not summary.samples:
                raise ValueError(f'Empty evaluation threshold: {threshold}')
            result = dict(
                summary.asDict(), split=split, threshold_mm=threshold,
                positive_rate=summary.positives / summary.samples
            )
            for metric in ('areaUnderROC', 'areaUnderPR'):
                result[metric] = (
                    BinaryClassificationEvaluator(
                        labelCol='label', rawPredictionCol='p',
                        metricName=metric, numBins=1000
                    ).evaluate(part)
                    if 0 < summary.positives < summary.samples else None
                )
            rows.append(result)
            print(result, flush=True)

        reliability = (
            scored.withColumn('bin', F.least(F.floor(F.col('p') * 10), F.lit(9)))
            .groupBy('threshold_mm', 'bin')
            .agg(
                F.count('*').alias('samples'),
                F.avg('p').alias('mean_probability'),
                F.avg('label').alias('observed_rate')
            )
            .orderBy('threshold_mm', 'bin')
            .collect()
        )
        return rows, [dict(r.asDict(), split=split) for r in reliability]
    finally:
        scored.unpersist()

def write_csv(path, rows):
    """Write small collected summaries; full predictions remain in Parquet."""
    if not rows:
        raise ValueError(f'No rows to write: {path}')
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def read_dataset_split(spark, dataset_root, name):
    """Read shared prepared observations or data_prep threshold examples."""
    if name not in ('train', 'validation', 'test'):
        raise ValueError(f'Unknown split: {name}')
    dataset = Path(dataset_root).resolve()
    examples = (dataset / 'examples').is_dir()
    base = dataset / 'examples' if examples else dataset
    frame = spark.read.option('basePath', str(base)).parquet(str(base / f'split={name}'))
    if examples:
        aliases = {'timestamp': 'prediction_time', 'past_5m_mm': 'rainfall_previous_5m',
                   'past_30m_mm': 'rainfall_previous_30m', 'past_60m_mm': 'rainfall_previous_60m'}
        for old, new in aliases.items():
            if old in frame.columns:
                frame = frame.withColumnRenamed(old, new)
        required = [c for c in REQUIRED_COLUMNS if c != 'future_rainfall_30m'] + ['threshold_mm', 'label']
    else:
        required = REQUIRED_COLUMNS
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f'Missing columns in {name}: {sorted(missing)}')
    frame = frame.dropna(subset=required)
    for column in required:
        if column in RAINFALL_COLUMNS or column in ('label', 'threshold_mm'):
            frame = frame.where((~F.isnan(F.col(column))) & (F.abs(F.col(column)) != float('inf')))
    if examples:
        if frame.where((~F.col('label').isin(0., 1.)) | (~F.col('threshold_mm').isin(THRESHOLDS))).limit(1).count():
            raise ValueError('Invalid labels or thresholds')
    else:
        frame = frame.withColumn('threshold_mm', F.explode(F.array(*[F.lit(t) for t in THRESHOLDS])))
        frame = frame.withColumn('label', (F.col('future_rainfall_30m') > F.col('threshold_mm')).cast('double'))
    return frame.select('station_id', 'prediction_time', 'hour', 'month',
                        'rainfall_previous_5m', 'rainfall_previous_30m',
                        'rainfall_previous_60m', 'threshold_mm', 'label')

In [4]:
def run_testing(spark, dataset_root, run_root, decision_threshold=0.5):
    """Score a frozen pipeline and save metrics, provenance and predictions."""
    if not 0 < decision_threshold < 1:
        raise ValueError('decision_threshold must be in (0, 1)')
    run = Path(run_root).resolve()
    manifest = json.loads((run / 'run_manifest.json').read_text(encoding='utf-8'))
    if manifest.get('status') != 'complete':
        raise ValueError('Select a completed training run')
    if Path(manifest['dataset_root']).resolve() != Path(dataset_root).resolve():
        raise ValueError('Test dataset must match the training dataset root')
    output = run / 'test_evaluation' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output.mkdir(parents=True, exist_ok=False)
    record = dict(status='running', model_run=str(run), dataset_root=str(Path(dataset_root).resolve()),
                  decision_threshold=decision_threshold, split='test')
    def save():
        (output / 'test_manifest.json').write_text(json.dumps(record, indent=2), encoding='utf-8')
    save()
    scored = None
    try:
        model = PipelineModel.load(str(run / 'model'))
        scored = (model.transform(read_dataset_split(spark, dataset_root, 'test'))
                  .withColumn('p', vector_to_array('probability')[1])
                  .withColumn('predicted_label', (F.col('p') >= decision_threshold).cast('double'))
                  .persist(StorageLevel.DISK_ONLY))
        metrics, bins = evaluate(scored, 'test')
        balances = {float(r['threshold_mm']): r['positives'] / r['samples']
                    for r in manifest['training_balance']}
        for row in metrics:
            q = balances[row['threshold_mm']]
            rate = row['positive_rate']
            # Evaluate the constant training-frequency forecast on test labels.
            baseline = rate * (1-q)**2 + (1-rate) * q**2
            row.update(baseline_brier=baseline,
                       brier_skill=1-row['brier']/baseline if baseline else None)
        write_csv(output / 'test_reliability_bins.csv', bins)
        confusion = (scored.groupBy('threshold_mm', 'label', 'predicted_label')
                     .count().orderBy('threshold_mm', 'label', 'predicted_label').collect())
        counts = {(float(r.threshold_mm), int(r.label), int(r.predicted_label)): r['count']
                  for r in confusion}
        write_csv(output / 'confusion_matrix.csv', [
            dict(threshold_mm=t, label=y, predicted_label=p, count=counts.get((t, y, p), 0))
            for t in sorted(balances) for y in (0, 1) for p in (0, 1)])
        for row in metrics:
            t = row['threshold_mm']
            tn, fp, fn, tp = [counts.get((t, y, p), 0)
                              for y, p in ((0, 0), (0, 1), (1, 0), (1, 1))]
            row.update(true_negative=tn, false_positive=fp, false_negative=fn, true_positive=tp,
                       precision=tp / (tp + fp) if tp + fp else None,
                       recall=tp / (tp + fn) if tp + fn else None,
                       f1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None,
                       decision_threshold=decision_threshold)
        write_csv(output / 'test_metrics.csv', metrics)
        # Compare the frozen model's saved validation metrics with its full test results.
        with (run / 'metrics.csv').open(encoding='utf-8', newline='') as handle:
            validation = {float(r['threshold_mm']): r for r in csv.DictReader(handle)
                          if r['split'] == 'validation'}
        comparison = []
        for row in metrics:
            previous = validation.get(row['threshold_mm'], {})
            item = dict(threshold_mm=row['threshold_mm'])
            for key in ('samples', 'positive_rate', 'areaUnderROC', 'areaUnderPR', 'brier'):
                item['validation_' + key] = previous.get(key)
                item['test_' + key] = row[key]
            comparison.append(item)
        write_csv(output / 'validation_test_comparison.csv', comparison)
        if manifest.get('feature_importance'):
            write_csv(output / 'feature_importance.csv', [
                dict(feature=name, importance=value)
                for name, value in sorted(manifest['feature_importance'].items(),
                                          key=lambda pair: pair[1], reverse=True)])
        # Full comparisons stay distributed; only a deterministic preview is collected.
        columns = ['station_id', 'prediction_time', 'threshold_mm', 'label', 'p', 'predicted_label']
        scored.select(*columns).write.parquet(str(output / 'predictions'))
        preview = scored.select(*columns).orderBy('prediction_time', 'station_id', 'threshold_mm').limit(200).collect()
        write_csv(output / 'prediction_preview.csv', [r.asDict() for r in preview])
        record.update(status='complete', metrics=metrics,
                      train_fraction=manifest.get('train_fraction'),
                      validation_fraction=manifest.get('validation_fraction'),
                      model_parameters={k: manifest.get(k) for k in ('max_iter', 'max_depth', 'step_size')})
        save()
        return output
    except Exception as exc:
        record.update(status='failed', error=str(exc))
        save()
        raise
    finally:
        if scored is not None:
            scored.unpersist()

def resolve_gbt05(runs_root):
    """Select by saved parameters, never by test scores or latest modification time."""
    candidates = []
    for path in Path(runs_root).resolve().rglob('run_manifest.json'):
        record = json.loads(path.read_text(encoding='utf-8'))
        if (record.get('status') == 'complete' and record.get('max_iter') == 40
                and record.get('max_depth') == 5 and record.get('step_size') == 0.1
                and (path.parent / 'model').is_dir()):
            candidates.append(path.parent)
    if len(candidates) != 1:
        found = '\n'.join(str(p) for p in candidates) or '(none)'
        raise ValueError('Expected one saved GBT-05 model (40 iterations, depth 5, step 0.1). '
                         'Set RUN_ROOT to select explicitly. Candidates:\n' + found)
    return candidates[0]

## Run the frozen model on the complete test split
Run once after validation selection. Re-executing creates a new output directory. Do not select parameters based on test results.

In [5]:
if not 0 < DECISION_THRESHOLD < 1 or CORES < 1:
    raise ValueError('Invalid decision threshold or core count')
if RUN_ROOT is None:
    if MODEL_ROOT is None:
        raise FileNotFoundError('Set MODEL_ROOT to the remote Model_GBT directory')
    run = resolve_gbt05(Path(MODEL_ROOT) / 'outputs' / 'tuning_stage1')
else:
    run = Path(RUN_ROOT).resolve()
manifest = json.loads((run / 'run_manifest.json').read_text(encoding='utf-8'))
if (manifest.get('status') != 'complete' or
        (manifest.get('max_iter'), manifest.get('max_depth'), manifest.get('step_size')) != (40, 5, 0.1)):
    raise ValueError('Select completed GBT-05: iterations=40, depth=5, step=0.1')
if not (run / 'model').is_dir() or not (run / 'metrics.csv').is_file():
    raise FileNotFoundError('Selected run requires model/ and metrics.csv')
dataset = Path(manifest['dataset_root'])
if not dataset.is_dir():
    raise FileNotFoundError(f'Training dataset unavailable: {dataset}')
print('Frozen model:', run)
print('Training fraction:', manifest.get('train_fraction'))
print('Evaluating complete test split; no training.')
spark = (SparkSession.builder.master(f'local[{CORES}]').appName('GBT05FinalTest')
         .config('spark.driver.memory', DRIVER_MEMORY)
         .config('spark.sql.session.timeZone', 'Asia/Singapore')
         .config('spark.sql.shuffle.partitions', str(max(8, CORES * 4)))
         .getOrCreate())
try:
    TEST_OUTPUT = run_testing(spark, dataset, run, DECISION_THRESHOLD)
    print('TEST COMPLETE. Results:', TEST_OUTPUT)
finally:
    spark.stop()


Frozen model: /nfs/home/svu/le.pan/singapore-rainfall-analysis-and-prediction/task2/Model_GBT/outputs/tuning_stage1/GBT-05
Training fraction: 0.02
Evaluating complete test split; no training.


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/16 18:35:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/16 18:36:11 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
                                                                                

{'samples': 6335645, 'positives': 402998.0, 'brier': 0.03602805809655804, 'split': 'test', 'threshold_mm': 0.0, 'positive_rate': 0.06360804622102406, 'areaUnderROC': 0.8601993509289053, 'areaUnderPR': 0.6269783773588539}


{'samples': 6335645, 'positives': 162858.0, 'brier': 0.017659235731918032, 'split': 'test', 'threshold_mm': 1.0, 'positive_rate': 0.02570503871350115, 'areaUnderROC': 0.8514877479000146, 'areaUnderPR': 0.48332247497512126}


{'samples': 6335645, 'positives': 53092.0, 'brier': 0.007494671863548281, 'split': 'test', 'threshold_mm': 5.0, 'positive_rate': 0.008379888709042252, 'areaUnderROC': 0.8578518938006793, 'areaUnderPR': 0.2680651838722235}


{'samples': 6335645, 'positives': 24252.0, 'brier': 0.004049417930846353, 'split': 'test', 'threshold_mm': 10.0, 'positive_rate': 0.0038278659868095513, 'areaUnderROC': 0.8634661221294951, 'areaUnderPR': 0.17028447453372147}


TEST COMPLETE. Results: /nfs/home/svu/le.pan/singapore-rainfall-analysis-and-prediction/task2/Model_GBT/outputs/tuning_stage1/GBT-05/test_evaluation/20261016T103557030872Z


In [6]:
# Only runs after a successful test. Uses the standard library.
for filename in ['test_metrics.csv', 'validation_test_comparison.csv', 'feature_importance.csv']:
    path = TEST_OUTPUT / filename
    if path.is_file():
        print('\n' + filename)
        with path.open(encoding='utf-8', newline='') as handle:
            for row in csv.DictReader(handle):
                print(row)



test_metrics.csv
{'samples': '6335645', 'positives': '402998.0', 'brier': '0.03602805809655804', 'split': 'test', 'threshold_mm': '0.0', 'positive_rate': '0.06360804622102406', 'areaUnderROC': '0.8601993509289053', 'areaUnderPR': '0.6269783773588539', 'baseline_brier': '0.05957356936729776', 'brier_skill': '0.39523418725460424', 'true_negative': '5882154', 'false_positive': '50493', 'false_negative': '224969', 'true_positive': '178029', 'precision': '0.7790453435555439', 'recall': '0.44176149757567035', 'f1': '0.5638111223714213', 'decision_threshold': '0.5'}
{'samples': '6335645', 'positives': '162858.0', 'brier': '0.017659235731918032', 'split': 'test', 'threshold_mm': '1.0', 'positive_rate': '0.02570503871350115', 'areaUnderROC': '0.8514877479000146', 'areaUnderPR': '0.48332247497512126', 'baseline_brier': '0.025049483190083474', 'brier_skill': '0.29502594532932613', 'true_negative': '6148714', 'false_positive': '24073', 'false_negative': '106768', 'true_positive': '56090', 'precis

Results include test metrics, reliability bins, confusion counts, example predictions, full scored Parquet, validation/test comparison and saved feature importance when available. Confirm test_manifest.json has status complete. Calibration is evaluated, not fitted. If trained on 2% of training data, retain that qualification in the report.